# 11 Build and Query the GraphRAG Layer
This notebook turns final decisions, graph evidence, candidate audit data, and XAI explanations into retrieval documents, embeds them, retrieves relevant context for a project question, and uses an LLM to generate a grounded GraphRAG answer.

In [0]:
# Load graph, XAI, final decision, clean, and all-candidate tables.
from pyspark.sql import functions as F

nodes_table = "workspace.entity_resolution_project.company_er_graph_nodes"
edges_table = "workspace.entity_resolution_project.company_er_graph_edges"
xai_table = "workspace.entity_resolution_project.company_er_xai_explanations"

rag_docs_table = "workspace.entity_resolution_project.company_er_graphrag_documents"
rag_retrieval_table = "workspace.entity_resolution_project.company_er_graphrag_retrieval_results"
rag_answers_table = "workspace.entity_resolution_project.company_er_graphrag_answers"

final_df = spark.table("workspace.entity_resolution_project.company_er_final_decisions")
clean_df = spark.table("workspace.entity_resolution_project.company_er_clean")
all_candidate_df = spark.table("workspace.entity_resolution_project.company_er_all_candidate_decisions")

EMBEDDING_MODEL = "databricks-gte-large-en"
LLM_MODEL = "databricks-gpt-oss-120b"

nodes = spark.table(nodes_table)
edges = spark.table(edges_table)
xai = spark.table(xai_table)

In [0]:
# Define a helper for safely formatting numeric values as text.
def rounded_text(col_name, digits=3):
    return F.coalesce(
        F.round(
            F.expr(f"try_cast(nullif(trim(cast({col_name} as string)), '') as double)"),
            digits
        ).cast("string"),
        F.lit("")
    )

In [0]:
# Prepare source and target node lookup tables for graph context.
source_nodes = (
    nodes
    .select(
        F.col("node_id").alias("source_node_id"),
        F.col("node_type").alias("source_node_type"),
        F.col("label").alias("source_label")
    )
)

target_nodes = (
    nodes
    .select(
        F.col("node_id").alias("target_node_id"),
        F.col("node_type").alias("target_node_type"),
        F.col("label").alias("target_label")
    )
)

In [0]:
# Build XAI explanation documents for retrieval.
xai_docs = (
    xai
    .withColumn(
    "doc_id",
        F.concat(
            F.lit("xai:"),
            F.col("left_row_key").cast("string"),
            F.lit(":"),
            F.col("right_row_key").cast("string")
        )
    )
    .withColumn("doc_type", F.lit("XAI_EXPLANATION"))
    .withColumn(
        "context_text",
        F.concat_ws(
            "\n",
            F.lit("XAI_EXPLANATION: match decision, scores, confidence, and reasoning."),
            F.concat(F.lit("Input company: "), F.coalesce(F.col("left_company_name"), F.lit(""))),
            F.concat(F.lit("Candidate company: "), F.coalesce(F.col("right_company_name"), F.lit(""))),
            F.concat(F.lit("Final decision: "), F.coalesce(F.col("final_decision"), F.lit(""))),
            F.concat(F.lit("Decision source: "), F.coalesce(F.col("final_decision_source"), F.lit(""))),
            F.concat(F.lit("Final confidence: "), rounded_text("final_confidence")),
            F.concat(F.lit("Name similarity: "), rounded_text("name_similarity")),
            F.concat(F.lit("Semantic similarity: "), rounded_text("semantic_similarity")),
            F.concat(F.lit("Country match: "), F.col("country_match").cast("string")),
            F.concat(F.lit("City match: "), F.col("city_match").cast("string")),
            F.concat(F.lit("Primary score driver: "), F.coalesce(F.col("primary_score_driver"), F.lit(""))),
            F.concat(F.lit("LLM decision: "), F.coalesce(F.col("llm_decision"), F.lit("none"))),
            F.concat(F.lit("LLM reason: "), F.coalesce(F.col("llm_reason"), F.lit("none"))),
            F.concat(F.lit("Final reason: "), F.coalesce(F.col("final_reason"), F.lit("")))
        )
    )
    .select("doc_id", "doc_type", "context_text")
)

In [0]:
# Build graph evidence text for each final decision pair.
graph_edge_context = (
    edges
    .join(source_nodes, on="source_node_id", how="left")
    .join(target_nodes, on="target_node_id", how="left")
    .withColumn(
        "edge_line",
        F.concat_ws(
            "",
            F.lit("- "),
            F.coalesce(F.col("source_label"), F.lit("")),
            F.lit(" -> "),
            F.coalesce(F.col("edge_type"), F.lit("")),
            F.lit(" -> "),
            F.coalesce(F.col("target_label"), F.lit("")),
            F.lit("; weight="),
            rounded_text("weight"),
            F.lit("; evidence="),
            F.coalesce(F.col("evidence"), F.lit(""))
        )
    )
    .select("source_node_id", "target_node_id", "edge_type", "edge_line")
)

final_graph_keys = (
    final_df
    .withColumn("input_node_id", F.concat(F.lit("input:"), F.col("left_row_key").cast("string")))
    .withColumn("candidate_node_id", F.concat(F.lit("company:"), F.col("right_row_key").cast("string")))
    .select("left_row_key", "right_row_key", "input_node_id", "candidate_node_id")
)

graph_context_by_pair = (
    final_graph_keys.alias("k")
    .join(
        graph_edge_context.alias("e"),
        (
            ((F.col("k.input_node_id") == F.col("e.source_node_id")) &
             (F.col("k.candidate_node_id") == F.col("e.target_node_id")))
            |
            (F.col("k.candidate_node_id") == F.col("e.source_node_id"))
        ),
        "left"
    )
    .filter(F.col("edge_line").isNotNull())
    .groupBy("left_row_key", "right_row_key")
    .agg(F.concat_ws("\n", F.sort_array(F.collect_set("edge_line"))).alias("graph_evidence"))
)

In [0]:
# Build exact company lookup documents across input and candidate roles.
company_as_input = (
    all_candidate_df
    .withColumn("lookup_company_name", F.col("left_company_name"))
    .withColumn("lookup_company_role", F.lit("INPUT_COMPANY"))
    .withColumn(
        "lookup_line",
        F.concat_ws(
            "",
            F.lit("- Appears as INPUT_COMPANY: "),
            F.coalesce(F.col("left_company_name"), F.lit("")),
            F.lit("; input country: "),
            F.coalesce(F.col("left_country"), F.lit("")),
            F.lit("; input country code: "),
            F.coalesce(F.col("left_country_code"), F.lit("")),
            F.lit("; input city: "),
            F.coalesce(F.col("left_city"), F.lit("")),
            F.lit("; candidate company: "),
            F.coalesce(F.col("right_company_name"), F.lit("")),
            F.lit("; candidate rank: "),
            F.coalesce(F.col("candidate_rank").cast("string"), F.lit("")),
            F.lit("; selected final candidate: "),
            F.coalesce(F.col("is_selected_final_candidate").cast("string"), F.lit("")),
            F.lit("; decision: "),
            F.coalesce(F.col("final_decision"), F.lit("")),
            F.lit("; confidence: "),
            rounded_text("final_confidence"),
            F.lit("; reason: "),
            F.coalesce(F.col("selection_reason"), F.col("final_reason"), F.lit(""))
        )
    )
    .select("lookup_company_name", "lookup_company_role", "lookup_line")
)

company_as_candidate = (
    all_candidate_df
    .withColumn("lookup_company_name", F.col("right_company_name"))
    .withColumn("lookup_company_role", F.lit("CANDIDATE_COMPANY"))
    .withColumn(
        "lookup_line",
        F.concat_ws(
            "",
            F.lit("- Appears as CANDIDATE_COMPANY: "),
            F.coalesce(F.col("right_company_name"), F.lit("")),
            F.lit("; candidate country: "),
            F.coalesce(F.col("candidate_country"), F.col("right_country"), F.lit("")),
            F.lit("; candidate country code: "),
            F.coalesce(F.col("candidate_country_code"), F.col("right_country_code"), F.lit("")),
            F.lit("; candidate city: "),
            F.coalesce(F.col("candidate_city"), F.col("right_city"), F.lit("")),
            F.lit("; candidate website URL: "),
            F.coalesce(F.col("candidate_website_url"), F.lit("")),
            F.lit("; candidate website domain: "),
            F.coalesce(F.col("candidate_website_domain"), F.lit("")),
            F.lit("; candidate commercial names: "),
            F.coalesce(F.col("candidate_commercial_names"), F.lit("")),
            F.lit("; candidate email: "),
            F.coalesce(F.col("candidate_email"), F.lit("")),
            F.lit("; candidate phone: "),
            F.coalesce(F.col("candidate_phone"), F.lit("")),
            F.lit("; input company: "),
            F.coalesce(F.col("left_company_name"), F.lit("")),
            F.lit("; candidate rank: "),
            F.coalesce(F.col("candidate_rank").cast("string"), F.lit("")),
            F.lit("; selected final candidate: "),
            F.coalesce(F.col("is_selected_final_candidate").cast("string"), F.lit("")),
            F.lit("; decision: "),
            F.coalesce(F.col("final_decision"), F.lit("")),
            F.lit("; confidence: "),
            rounded_text("final_confidence"),
            F.lit("; name similarity: "),
            rounded_text("name_similarity"),
            F.lit("; semantic similarity: "),
            rounded_text("semantic_similarity"),
            F.lit("; country match: "),
            F.coalesce(F.col("country_match").cast("string"), F.lit("")),
            F.lit("; city match: "),
            F.coalesce(F.col("city_match").cast("string"), F.lit("")),
            F.lit("; reason: "),
            F.coalesce(F.col("selection_reason"), F.col("final_reason"), F.lit(""))
        )
    )
    .select("lookup_company_name", "lookup_company_role", "lookup_line")
)

company_lookup_docs = (
    company_as_input
    .unionByName(company_as_candidate)
    .filter(F.col("lookup_company_name").isNotNull())
    .withColumn("lookup_company_key", F.lower(F.trim(F.col("lookup_company_name"))))
    .groupBy("lookup_company_key")
    .agg(
        F.collect_set("lookup_company_name").alias("company_name_variants"),
        F.collect_set("lookup_company_role").alias("company_roles"),
        F.concat_ws("\n", F.sort_array(F.collect_set("lookup_line"))).alias("lookup_lines")
    )
    .withColumn("doc_id", F.concat(F.lit("company_lookup:"), F.sha2(F.col("lookup_company_key"), 256)))
    .withColumn("doc_type", F.lit("COMPANY_LOOKUP"))
    .withColumn(
        "context_text",
        F.concat_ws(
            "\n",
            F.lit("COMPANY_LOOKUP: exact company lookup across input and candidate roles."),
            F.concat(F.lit("Company name key: "), F.col("lookup_company_key")),
            F.concat(F.lit("Company name variants: "), F.concat_ws(", ", F.col("company_name_variants"))),
            F.concat(F.lit("Roles found: "), F.concat_ws(", ", F.col("company_roles"))),
            F.lit("Use this document first for company lookup, match, no-match, candidate, and final-selection questions."),
            F.lit("Role-specific evidence:"),
            F.col("lookup_lines")
        )
    )
    .select("doc_id", "doc_type", "context_text")
)

In [0]:
# Build final graph decision documents with profile and graph evidence.
clean_profile_df = clean_df.select(
    F.col("unique_id").cast("string").alias("clean_unique_id"),
    F.col("main_country").cast("string").alias("profile_country"),
    F.col("main_country_code").cast("string").alias("profile_country_code"),
    F.col("main_city").cast("string").alias("profile_city"),
    F.col("website_url").cast("string").alias("profile_website_url"),
    F.col("website_domain").cast("string").alias("profile_website_domain"),
    F.col("company_commercial_names").cast("string").alias("profile_commercial_names"),
    F.col("primary_email").cast("string").alias("profile_email"),
    F.col("primary_phone").cast("string").alias("profile_phone"),
)

final_graph_decision_docs = (
    final_df
    .join(graph_context_by_pair, on=["left_row_key", "right_row_key"], how="left")
    .join(
        clean_profile_df,
        F.col("right_row_key").cast("string") == F.col("clean_unique_id"),
        "left"
    )
    .withColumn(
        "doc_id",
        F.concat(
            F.lit("final_graph_decision:"),
            F.col("left_row_key").cast("string"),
            F.lit(":"),
            F.col("right_row_key").cast("string")
        )
    )
    .withColumn("doc_type", F.lit("FINAL_GRAPH_DECISION"))
    .withColumn(
        "context_text",
        F.concat_ws(
            "\n",
            F.lit("FINAL_GRAPH_DECISION: top-1 final decision grounded in knowledge graph nodes and edges."),
            F.concat(F.lit("Input company: "), F.coalesce(F.col("left_company_name"), F.lit(""))),
            F.concat(F.lit("Final candidate company: "), F.coalesce(F.col("right_company_name"), F.lit(""))),
            F.concat(F.lit("Input country: "), F.coalesce(F.col("left_country"), F.lit(""))),
            F.concat(F.lit("Input country code: "), F.coalesce(F.col("left_country_code"), F.lit(""))),
            F.concat(F.lit("Candidate country: "), F.coalesce(F.col("right_country"), F.col("profile_country"), F.lit(""))),
            F.concat(F.lit("Candidate country code: "), F.coalesce(F.col("right_country_code"), F.col("profile_country_code"), F.lit(""))),
            F.concat(F.lit("Candidate city: "), F.coalesce(F.col("right_city"), F.col("profile_city"), F.lit(""))),
            F.concat(F.lit("Candidate commercial names: "), F.coalesce(F.col("profile_commercial_names"), F.lit(""))),
            F.concat(F.lit("Candidate website URL: "), F.coalesce(F.col("profile_website_url"), F.lit(""))),
            F.concat(F.lit("Candidate website domain: "), F.coalesce(F.col("profile_website_domain"), F.lit(""))),
            F.concat(F.lit("Candidate email: "), F.coalesce(F.col("profile_email"), F.lit(""))),
            F.concat(F.lit("Candidate phone: "), F.coalesce(F.col("profile_phone"), F.lit(""))),
            F.concat(F.lit("Final decision: "), F.coalesce(F.col("final_decision"), F.lit(""))),
            F.concat(F.lit("Final confidence: "), rounded_text("final_confidence")),
            F.concat(F.lit("Final reason: "), F.coalesce(F.col("final_reason"), F.lit(""))),
            F.lit("Knowledge graph evidence:"),
            F.coalesce(F.col("graph_evidence"), F.lit("not available"))
        )
    )
    .select("doc_id", "doc_type", "context_text")
)

In [0]:
# Build documents for non-top candidates that were considered but dropped.
non_top_candidate_docs = (
    all_candidate_df
    .filter(F.col("candidate_rank") > 1)
    .withColumn(
        "doc_id",
        F.concat(
            F.lit("non_top_candidate:"),
            F.col("left_row_key").cast("string"),
            F.lit(":"),
            F.col("right_row_key").cast("string")
        )
    )
    .withColumn("doc_type", F.lit("NON_TOP_CANDIDATE_DECISION"))
    .withColumn(
        "context_text",
        F.concat_ws(
            "\n",
            F.lit("NON_TOP_CANDIDATE_DECISION: candidate pair that was considered but not selected as the final top-1 candidate."),
            F.concat(F.lit("Input company: "), F.coalesce(F.col("left_company_name"), F.lit(""))),
            F.concat(F.lit("Candidate company: "), F.coalesce(F.col("right_company_name"), F.lit(""))),
            F.concat(F.lit("Input country: "), F.coalesce(F.col("left_country"), F.lit(""))),
            F.concat(F.lit("Input country code: "), F.coalesce(F.col("left_country_code"), F.lit(""))),
            F.concat(F.lit("Input city: "), F.coalesce(F.col("left_city"), F.lit(""))),
            F.concat(F.lit("Candidate country: "), F.coalesce(F.col("candidate_country"), F.col("right_country"), F.lit(""))),
            F.concat(F.lit("Candidate country code: "), F.coalesce(F.col("candidate_country_code"), F.col("right_country_code"), F.lit(""))),
            F.concat(F.lit("Candidate city: "), F.coalesce(F.col("candidate_city"), F.col("right_city"), F.lit(""))),
            F.concat(F.lit("Candidate website URL: "), F.coalesce(F.col("candidate_website_url"), F.lit(""))),
            F.concat(F.lit("Candidate website domain: "), F.coalesce(F.col("candidate_website_domain"), F.lit(""))),
            F.concat(F.lit("Candidate commercial names: "), F.coalesce(F.col("candidate_commercial_names"), F.lit(""))),
            F.concat(F.lit("Candidate email: "), F.coalesce(F.col("candidate_email"), F.lit(""))),
            F.concat(F.lit("Candidate phone: "), F.coalesce(F.col("candidate_phone"), F.lit(""))),
            F.concat(F.lit("Candidate rank: "), F.col("candidate_rank").cast("string")),
            F.concat(F.lit("Selected final candidate: "), F.col("is_selected_final_candidate").cast("string")),
            F.concat(F.lit("Final selection status: "), F.coalesce(F.col("final_selection_status"), F.lit(""))),
            F.concat(F.lit("Dropped stage: "), F.coalesce(F.col("dropped_stage"), F.lit(""))),
            F.concat(F.lit("Pair decision: "), F.coalesce(F.col("final_decision"), F.lit(""))),
            F.concat(F.lit("Decision source: "), F.coalesce(F.col("final_decision_source"), F.lit(""))),
            F.concat(F.lit("Confidence: "), rounded_text("final_confidence")),
            F.concat(F.lit("Name similarity: "), rounded_text("name_similarity")),
            F.concat(F.lit("Semantic similarity: "), rounded_text("semantic_similarity")),
            F.concat(F.lit("Country match: "), F.coalesce(F.col("country_match").cast("string"), F.lit(""))),
            F.concat(F.lit("City match: "), F.coalesce(F.col("city_match").cast("string"), F.lit(""))),
            F.concat(F.lit("Selection reason: "), F.coalesce(F.col("selection_reason"), F.lit(""))),
            F.concat(F.lit("Final reason: "), F.coalesce(F.col("final_reason"), F.lit("")))
        )
    )
    .select("doc_id", "doc_type", "context_text")
)

In [0]:
# Create a structured project summary for aggregate GraphRAG context.
# Final decision counts
decision_counts = final_df.groupBy("final_decision").count().collect()
decision_count_text = "\n".join([
    f"- {row['final_decision']} decisions count: {row['count']}"
    for row in decision_counts
])

# Decision source counts
source_counts = final_df.groupBy("final_decision_source").count().collect()
source_count_text = "\n".join([
    f"- {row['final_decision_source']}: {row['count']}"
    for row in source_counts
])

# MATCH suppliers by country
country_rows = (
    final_df
    .filter(F.col("final_decision") == "MATCH")
    .groupBy("right_country_code")
    .agg(
        F.countDistinct("right_company_name").alias("resolved_suppliers"),
        F.collect_set("right_company_name").alias("supplier_names")
    )
    .orderBy(F.desc("resolved_suppliers"))
    .collect()
)

country_text = "\n".join([
    f"- {row['right_country_code']}: {row['resolved_suppliers']} suppliers ({', '.join(row['supplier_names'][:10])})"
    for row in country_rows
])

# MATCH suppliers by city
city_rows = (
    final_df
    .filter(F.col("final_decision") == "MATCH")
    .groupBy("right_country_code", "right_city")
    .agg(
        F.countDistinct("right_company_name").alias("resolved_suppliers"),
        F.collect_set("right_company_name").alias("supplier_names")
    )
    .orderBy(F.desc("resolved_suppliers"))
    .collect()
)

city_text = "\n".join([
    f"- {row['right_city']}, {row['right_country_code']}: {row['resolved_suppliers']} suppliers ({', '.join(row['supplier_names'][:8])})"
    for row in city_rows
])

# Suppliers matched from multiple input names
multi_name_rows = (
    final_df
    .filter(F.col("final_decision") == "MATCH")
    .groupBy("right_company_name")
    .agg(
        F.countDistinct("left_company_name").alias("distinct_input_names"),
        F.collect_set("left_company_name").alias("input_names"),
        F.count("*").alias("match_rows")
    )
    .filter(F.col("distinct_input_names") > 1)
    .orderBy(F.desc("distinct_input_names"), F.desc("match_rows"))
    .collect()
)

multi_name_text = "\n".join([
    f"- {row['right_company_name']}: {row['distinct_input_names']} input names ({', '.join(row['input_names'])})"
    for row in multi_name_rows
])

# Review cases
review_rows = (
    final_df
    .filter(F.col("final_decision") == "REVIEW")
    .select(
        "left_company_name",
        "right_company_name",
        "right_country_code",
        "right_city",
        "final_confidence",
        "final_reason"
    )
    .orderBy(F.desc("final_confidence"))
    .collect()
)

review_text = "\n".join([
    f"- {row['left_company_name']} -> {row['right_company_name']}: country={row['right_country_code']}, city={row['right_city']}, confidence={row['final_confidence']}, reason={row['final_reason']}"
    for row in review_rows
])

structured_summary = f"""
Project summary:
- Total input records: {final_df.count()}

Final decision counts:
{decision_count_text}

Decision source counts:
{source_count_text}

Resolved MATCH suppliers by country:
{country_text}

Resolved MATCH suppliers by city:
{city_text}

Suppliers matched from multiple input names:
{multi_name_text}

Records needing review:
{review_text}
"""

print(structured_summary)

In [0]:
# Convert the structured project summary into a retrieval document.
summary_docs = spark.createDataFrame(
    [
        (
            "summary:structured_project_summary",
            "PROJECT_SUMMARY",
            f"""
            PROJECT_SUMMARY

            This document contains the full structured project summary.
            It includes:
            - final decision counts
            - MATCH decisions count
            - REVIEW decisions count
            - NO_MATCH decisions count
            - decision source counts
            - suppliers matched from multiple input names
            - supplier alias groups
            - suppliers by country
            - suppliers by city
            - records needing review

            {structured_summary}
            """
        )
    ],
    ["doc_id", "doc_type", "context_text"]
)

In [0]:
# Combine all GraphRAG document types and save the document table.
rag_docs = (
    final_graph_decision_docs
    .unionByName(non_top_candidate_docs)
    .unionByName(xai_docs)
    .unionByName(company_lookup_docs)
    .unionByName(summary_docs)
    .dropDuplicates(["doc_id"])
)

(
    rag_docs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(rag_docs_table)
)

print("GraphRAG documents:", rag_docs.count())
display(rag_docs.groupBy("doc_type").count())
display(rag_docs)

In [0]:
# Define the user question, retrieval size, and answer rules.
QUESTION = "Which companies were candidate for the company Amazon E Store and which company matched to it and why was the reason, i want all of its candidate to be listed"
TOP_K = 50
ANSWER_RULES = """
Write the answer in a natural project-report style.

CRITICAL RULES:
- Use only the provided project context. Do not invent facts.
- Treat each user message as standalone.
- Answer only the latest user question.
- Do not reuse company names, countries, retrieved documents, or answers from previous turns.
- For every new question, retrieve using only the latest user question.
- For specific company questions, never search for a company name that is not explicitly present in the latest user question.
- For specific company questions, if retrieved documents do not mention the latest target company, say it was not found in the available project context.

Source priority:
- Use COMPANY_LOOKUP first for any specific company name. It shows whether the company appears as an input company, candidate company, or both.
- Use FINAL_GRAPH_DECISION for selected top-1 final supplier matches.
- Use NON_TOP_CANDIDATE_DECISION for candidates that were considered but not selected as final.
- Use XAI_EXPLANATION for score interpretation, confidence, LLM reason, and final reason.
- Use PROJECT_SUMMARY for counts, totals, distributions, grouped findings, and alias summaries.

Company lookup logic:
- A company may appear as an input company, candidate company, final matched supplier, or multiple roles.
- Check all retrieved roles before saying NOT_FOUND.
- If the company appears only as a candidate, start with "CANDIDATE: ...".
- If the company appears as an input with a final MATCH, start with "MATCH: ...".
- If the company appears as an input with REVIEW, start with "REVIEW: ...".
- If the company appears as an input/candidate pair with NO_MATCH, start with "NO_MATCH: ...".
- If the company is not found in any retrieved role, start with "NOT_FOUND: ...".
- For candidate questions, list all retrieved candidate pairs with input company, candidate company, rank, decision, confidence, and reason when available.
- If a candidate was not selected as final, explain that only the top-ranked candidate is kept in company_er_final_decisions.

Location and supplier fields:
- Include city, country, country code, website URL, website domain, email, and phone when available.
- If the question asks about companies in a country/city, clearly separate final matched suppliers from non-top candidates.
- Do not imply the list is complete unless PROJECT_SUMMARY provides a complete total.
- If a field is missing, say "not available in the dataset".

Reasoning:
- If asked why a decision was made, explain relevant rank, score, confidence, country/city match, LLM reason, and final reason.
- Translate graph evidence into natural language.
- Do not mention raw graph edge names such as HAS_TOP_CANDIDATE, MATCHES, LOCATED_IN, NEEDS_REVIEW, or NO_MATCH.
- Do not say "edge" unless the user explicitly asks about graph structure.

Answer style:
- Start with one short summary sentence.
- Use bullets after the first sentence.
- Do not use a markdown table.
- Use company names, not node IDs.
- Mention confidence or score only when useful.
- If evidence is incomplete, say manual inspection is needed.
- Keep the answer concise.
"""

In [0]:
# Generate embeddings for retrieval documents and the user question.
docs_for_embedding = (
    spark.table(rag_docs_table)
    .withColumn("embedding_text", F.substring(F.col("context_text"), 1, 6000))
)

doc_embeddings = docs_for_embedding.selectExpr(
    "*",
    f"ai_query('{EMBEDDING_MODEL}', embedding_text) AS doc_embedding"
)

query_df = spark.createDataFrame([(QUESTION,)], ["question"])

query_embedding = query_df.selectExpr(
    "*",
    f"ai_query('{EMBEDDING_MODEL}', question) AS query_embedding"
)

In [0]:
# Retrieve the most relevant GraphRAG documents by cosine similarity.
retrieved = (
    doc_embeddings
    .crossJoin(query_embedding)
    .withColumn(
        "semantic_dot",
        F.expr("""
            aggregate(
              zip_with(doc_embedding, query_embedding, (x, y) -> x * y),
              cast(0.0 as double),
              (acc, z) -> acc + z
            )
        """)
    )
    .withColumn(
        "doc_norm",
        F.sqrt(
            F.expr("""
                aggregate(
                  transform(doc_embedding, x -> x * x),
                  cast(0.0 as double),
                  (acc, z) -> acc + z
                )
            """)
        )
    )
    .withColumn(
        "query_norm",
        F.sqrt(
            F.expr("""
                aggregate(
                  transform(query_embedding, x -> x * x),
                  cast(0.0 as double),
                  (acc, z) -> acc + z
                )
            """)
        )
    )
    .withColumn(
        "retrieval_score",
        F.when(
            (F.col("doc_norm") > 0) & (F.col("query_norm") > 0),
            F.col("semantic_dot") / (F.col("doc_norm") * F.col("query_norm"))
        ).otherwise(F.lit(0.0))
    )
    .select("question", "doc_id", "doc_type", "context_text", "retrieval_score")
    .orderBy(F.desc("retrieval_score"))
    .limit(TOP_K)
)

(
    retrieved.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(rag_retrieval_table)
)

display(retrieved)

In [0]:
# Assemble retrieved documents into the prompt context.
retrieved_rows = retrieved.collect()

context_blocks = []
for i, row in enumerate(retrieved_rows, start=1):
    context_blocks.append(
        f"[Context {i} | {row['doc_type']} | score={row['retrieval_score']:.3f}]\n{row['context_text']}"
    )

rag_context = "\n\n".join(context_blocks)

In [0]:
# Build the grounded GraphRAG prompt.
rag_prompt = f"""
You are an entity resolution analyst writing for a project report.

Use only the information provided below.
Do not invent facts.
If the question is outside the entity resolution project, say that it cannot be answered from the available project context.

Question:
{QUESTION}

Structured project summary:
{structured_summary}

Retrieved graph and XAI context:
{rag_context}

Answer rules:
{ANSWER_RULES}

Answer:
"""

In [0]:
# Generate and save the GraphRAG answer.
prompt_df = spark.createDataFrame([(QUESTION, rag_prompt)], ["question", "rag_prompt"])

rag_answer = prompt_df.selectExpr(
    "*",
    f"""
    ai_query(
      '{LLM_MODEL}',
      rag_prompt,
      modelParameters => named_struct('temperature', 0.1),
      failOnError => false
    ) AS rag_raw_response
    """
)

rag_answer = (
    rag_answer
    .withColumn("answer_text", F.col("rag_raw_response.result"))
    .withColumn("error_message", F.col("rag_raw_response.errorMessage"))
)

(
    rag_answer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(rag_answers_table)
)

display(rag_answer.select("question", "answer_text", "error_message"))

In [0]:
# Review retrieved context used for the GraphRAG answer.
display(
    spark.table(rag_retrieval_table)
    .select("doc_type", "retrieval_score", "context_text")
    .orderBy(F.desc("retrieval_score"))
)